In [ ]:
# =========================
# 1. IMPORTS + SEED
# =========================
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

# =========================
# 2. MAZE GRIDWORLD
# =========================
class GridWorld:
    def __init__(self, size=6, obstacles=None, sparse=False, shaped=False):
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.state = self.start
        self.sparse = sparse
        self.shaped = shaped
        self.obstacles = obstacles if obstacles else []

    def reset(self):
        self.state = self.start
        return np.array(self.state, dtype=np.float32)

    def step(self, action):
        x, y = self.state

        if action == 0: x -= 1
        elif action == 1: x += 1
        elif action == 2: y -= 1
        elif action == 3: y += 1

        x = max(0, min(self.size - 1, x))
        y = max(0, min(self.size - 1, y))

        new_state = (x, y)

        if new_state in self.obstacles:
            new_state = self.state

        self.state = new_state

        # Distance to goal
        dist = abs(self.goal[0] - x) + abs(self.goal[1] - y)

        if self.state == self.goal:
            return np.array(self.state, dtype=np.float32), 10, True

        # Reward shaping
        if self.shaped:
            reward = -dist * 0.1
        elif self.sparse:
            reward = 0
        else:
            reward = -1

        return np.array(self.state, dtype=np.float32), reward, False


# =========================
# 3. MAZE DESIGN (IMPORTANT)
# =========================
maze_obstacles = [
    (1,1), (1,2), (1,3),
    (2,3),
    (3,1), (3,2),
    (4,4),
    (2,1), (4,2)   # NEW
]

# =========================
# 4. DQN MODEL
# =========================
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 4)
        )

    def forward(self, x):
        return self.net(x)

